# FastVLA: Ultra-Efficient VLA Fine-Tuning on Tesla T4

FastVLA is a high-performance framework designed to democratize Vision-Language-Action (VLA) models. This notebook is optimized for Kaggle 2x T4 GPUs, enabling you to fine-tune massive 7B models like OpenVLA on consumer-grade hardware.

### Performance Advantages
- **Distributed Training**: Automatically utilizes dual T4 GPUs for synchronized acceleration.
- **15x More Efficient**: Benchmark proves that FastVLA trains a 7B model with only 3.3x slowdown compared to a 135M model.
- **4-Bit QLoRA**: Reduces memory usage from 28GB+ to ~5GB, fitting perfectly on a T4.
- **Custom Triton Kernels**: Ultra-fast action prediction and fusion bypassing standard bottlenecks.

> **Task:** Fine-tune OpenVLA for the PushT robotics task (Image-to-Action) directly in this notebook.

## 🛠️ Step 0: Prerequisites & Setup

### 1. Gated Model Access (Llama-2)
OpenVLA-7B uses **Llama-2** as its backbone. You must request access to [meta-llama/Llama-2-7b-hf](https://huggingface.co/meta-llama/Llama-2-7b-hf) on HuggingFace.

### 2. Kaggle Secrets (HuggingFace Token)
To download gated models and push your results to the hub:
1. Go to **Add-ons -> Secrets** in the top menu.
2. Add a new secret with Label: `HF_TOKEN` and Value: `your_huggingface_write_token`.
3. Toggle **Internet** to **On** in the right sidebar settings.

In [ ]:
# ── 1. Deep Clean & Fresh Install ──────────────────────────────────────────
# Kaggle pre-installs incompatible versions of Numpy and Scipy.
# We wipe them completely to prevent potential binary conflicts.

print("Cleaning pre-installed packages...")
!pip uninstall -y numpy scipy pandas transformers torch torchvision torchaudio fastvla bitsandbytes accelerate peft datasets timm > /dev/null

print("Installing synchronized FastVLA stack...")
# We pin torch to a stable T4-compatible version to avoid CUDA 13.0 mismatch errors
!pip install -q --no-cache-dir \
    "numpy>=1.24.0,<2.0.0" \
    "scipy>=1.10.0" \
    "torch==2.4.1" \
    "torchvision==0.19.1" \
    "huggingface_hub>=0.30.0" \
    "git+https://github.com/unslothai/unsloth.git" \
    "fastvla[all] @ git+https://github.com/BouajilaHamza/FastVLA.git" \
    "bitsandbytes>=0.42.0" \
    "accelerate>=0.28.0" \
    "peft>=0.9.0" \
    "transformers>=4.40.0" \
    "datasets>=2.16.0" \
    "timm>=0.9.12"

print("\nInstallation Complete!")
print("ACTION REQUIRED: You MUST restart the session now.")
print("Go to 'Run' -> 'Restart Session' in the top menu.")

In [ ]:
# ── 2. Authentication & Health Check ───────────────────────────────────────
import os
import torch
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

try:
    token = UserSecretsClient().get_secret("HF_TOKEN")
    login(token=token)
    print("✅ Authenticated with Hugging Face")
except Exception as e:
    print("⚠️ Warning: HF_TOKEN secret not found. Gated models may fail to load.")

import fastvla
from fastvla import FastVLAModel, check_environment
from fastvla.kernels import TRITON_AVAILABLE

print(f"\n📦 FastVLA Version: {fastvla.__version__}")
print(f"🔥 GPU: {torch.cuda.get_device_name(0)} (x{torch.cuda.device_count()})")
print(f"⚡ Triton Kernels: {'✓ Active' if TRITON_AVAILABLE else 'ℹ Inactive (using PyTorch fallback)'}")
check_environment()

## 🤖 Step 1: Load Model (OpenVLA-7B in 4-bit)

We load the model with **4-bit quantization** and **LoRA** adapters enabled. This is the secret sauce that allows a 7B parameter model to fit on a 15GB T4 GPU.

**Configuration:**
- `model_id`: "openvla-7b" (or "smolvla" for a non-gated lightweight alternative).
- `action_dim`: **2** for PushT (x, y coordinates). Most other robotics datasets use 7.

In [ ]:
from fastvla import FastVLAModel

MODEL_ID = "openvla-7b"
ACTION_DIM = 2 # PushT uses 2D actions

print(f"📥 Loading {MODEL_ID} (4-bit + LoRA)...")

model = FastVLAModel.from_pretrained(
    MODEL_ID,
    load_in_4bit=True,
    use_peft=True,
    action_dim=ACTION_DIM,
    hf_token=token, # Pass the token for gated access (Llama-2)
    device_map="auto", # Automatically shards across 2x T4
    trust_remote_code=True
)

print(f"✅ Model loaded. Action Dimension: {model.config.action_dim}")

## 📊 Step 2: Prepare Dataset (PushT)

We use the `lerobot/pusht_image` dataset. FastVLA includes a built-in helper to fetch and process this into the correct VLA format (Images + Instructions -> Actions).

In [ ]:
from fastvla.data.datasets import get_dataset

print("📥 Loading PushT Dataset...")
dataset = get_dataset("lerobot/pusht_image")

print(f"✅ Dataset loaded. Total samples: {len(dataset)}")
print(f"📝 Sample Instruction: {dataset[0]['instructions']}")
print(f"📐 Action Shape: {dataset[0]['actions'].shape}")

## 🏋️ Step 3: Optimized Training

The `FastVLATrainer` handles the complexity of distributed training and mixed precision for you. 

**Optimizations for 2x T4:**
- `batch_size=4`: Good balance for 15GB VRAM.
- `gradient_accumulation_steps=2`: Effective batch size of 8 (4 per GPU * 2 acc).
- `use_8bit_optimizer=True`: Saves 2GB+ VRAM by quantizing optimizer states.
- `max_steps=500`: Enough to see significant progress on PushT.

In [ ]:
from fastvla import FastVLATrainer

trainer = FastVLATrainer(
    model=model,
    dataset=dataset,
    batch_size=4,
    lr=1e-4,
    max_steps=500, 
    gradient_accumulation_steps=2,
    use_8bit_optimizer=True,
    use_mixed_precision=True,
    output_dir="./fastvla-pusht-checkpoint",
    logging_steps=10,
    save_steps=250,
)

print("🔥 Starting Finetuning on 2x T4 GPUs...")
trainer.train()
print("✅ Training Complete!")

## 🔍 Step 4: Inference (Testing the Brain)

Let's see if the model can predict an action from a random image and prompt.

In [ ]:
import torch
from PIL import Image
import numpy as np

# 1. Create a dummy observation
dummy_img = Image.fromarray(np.random.randint(0, 255, (224, 224, 3), dtype=np.uint8))
instruction = "push the t-shaped block into the goal zone"

# 2. Process inputs
pixel_values = torch.randn(1, 1, 3, 224, 224).to(model.device)
input_ids = model.tokenizer(instruction, return_tensors="pt").input_ids.to(model.device)

# 3. Run Inference
model.eval()
with torch.no_grad():
    action_preds, _ = model(pixel_values=pixel_values, input_ids=input_ids)

print(f"🎯 Predicted Action (2D): {action_preds[0].cpu().numpy()}")

## 📤 Step 5: Export Model

You can now push your trained LoRA weights to the Hugging Face Hub or download them locally.

In [ ]:
# Example: Push to Hub
# repo_id = "your-username/FastVLA-PushT-OpenVLA"
# model.push_to_hub(repo_id, token=token)

print("🚀 Notebook completed successfully! Check the './fastvla-pusht-checkpoint' folder for weights.")

---
**Star the repo** to support democratized robotics!
[GitHub: FastVLA](https://github.com/BouajilaHamza/FastVLA)

**Created by the FastVLA Team.**